# 08 Human In The Loop

In [1]:
# Start coding here
# ==================================================
# Notebook 08
# Human-In-The-Loop
# ==================================================

from typing import TypedDict
from typing import List
from typing import Dict

from langgraph.graph import StateGraph, END

In [2]:
class ResearchState(TypedDict):

    research_goal: str

    queries_executed: List[str]

    raw_findings: List[str]

    deduplicated_findings: List[str]

    missing_topics: List[str]

    conflicts: List[Dict]

    critic_score: float

    trust_score: float

    draft_report: str

    final_report: str

    iteration_count: int

    approval_status: str

    reviewer_comments: str

    status: str

    errors: List[str]

    metadata: Dict

In [3]:
state = {
    "research_goal": "Compare cloud costs",
    "queries_executed": [],
    "raw_findings": [],
    "deduplicated_findings": [
        "AWS pricing increased 8%",
        "Azure pricing remained stable",
    ],
    "missing_topics": [],
    "conflicts": [],
    "critic_score": 0.82,
    "trust_score": 0.91,
    "draft_report": "",
    "final_report": "",
    "iteration_count": 1,
    "approval_status": "",
    "reviewer_comments": "",
    "status": "",
    "errors": [],
    "metadata": {},
}

In [4]:
def writer_agent(state):

    report = "\n".join(state["deduplicated_findings"])

    state["draft_report"] = report

    state["status"] = "draft_created"

    return state

In [5]:
state = writer_agent(state)

print(state["draft_report"])

AWS pricing increased 8%
Azure pricing remained stable


In [6]:
def human_review_agent(state):

    print("\nHuman Review Required")

    print("\nDraft Report:")

    print(state["draft_report"])

    return state

In [7]:
human_review_agent(state)


Human Review Required

Draft Report:
AWS pricing increased 8%
Azure pricing remained stable


{'research_goal': 'Compare cloud costs',
 'queries_executed': [],
 'raw_findings': [],
 'deduplicated_findings': ['AWS pricing increased 8%',
  'Azure pricing remained stable'],
 'missing_topics': [],
 'conflicts': [],
 'critic_score': 0.82,
 'trust_score': 0.91,
 'draft_report': 'AWS pricing increased 8%\nAzure pricing remained stable',
 'final_report': '',
 'iteration_count': 1,
 'approval_status': '',
 'reviewer_comments': '',
 'status': 'draft_created',
 'errors': [],
 'metadata': {}}

In [8]:
def approve_report(state):

    state["approval_status"] = "approved"

    state["reviewer_comments"] = "Looks good"

    return state

In [9]:
def reject_report(state):

    state["approval_status"] = "rejected"

    state["reviewer_comments"] = "Need competitor data"

    return state

In [10]:
state = approve_report(state)

state["approval_status"]

'approved'

In [11]:
def approval_router(state):

    if state["approval_status"] == "approved":

        return "publish"

    return "research"

In [12]:
def publish_agent(state):

    state["final_report"] = state["draft_report"]

    state["status"] = "published"

    return state

In [13]:
def research_retry_agent(state):

    state["iteration_count"] += 1

    state["status"] = "research_retry"

    return state

In [14]:
graph = StateGraph(ResearchState)

In [15]:
graph.add_node("writer", writer_agent)

graph.add_node("human_review", human_review_agent)

graph.add_node("publish", publish_agent)

graph.add_node("research", research_retry_agent)

In [16]:
graph.set_entry_point("writer")

graph.add_edge("writer", "human_review")

In [17]:
graph.add_conditional_edges("human_review", approval_router)

In [18]:
graph.add_edge("publish", END)

graph.add_edge("research", END)

In [19]:
app = graph.compile()

In [20]:
approved_state = state.copy()

approved_state["approval_status"] = "approved"

result = app.invoke(approved_state)

result["status"]


Human Review Required

Draft Report:
AWS pricing increased 8%
Azure pricing remained stable


'published'

In [21]:
rejected_state = state.copy()

rejected_state["approval_status"] = "rejected"

result = app.invoke(rejected_state)

result["status"]


Human Review Required

Draft Report:
AWS pricing increased 8%
Azure pricing remained stable


'research_retry'

In [22]:
checkpoint_store = []

In [23]:
def save_checkpoint(state):

    checkpoint_store.append(state.copy())

In [24]:
save_checkpoint(state)

len(checkpoint_store)

1

In [25]:
restored_state = checkpoint_store[0]

restored_state

{'research_goal': 'Compare cloud costs',
 'queries_executed': [],
 'raw_findings': [],
 'deduplicated_findings': ['AWS pricing increased 8%',
  'Azure pricing remained stable'],
 'missing_topics': [],
 'conflicts': [],
 'critic_score': 0.82,
 'trust_score': 0.91,
 'draft_report': 'AWS pricing increased 8%\nAzure pricing remained stable',
 'final_report': '',
 'iteration_count': 1,
 'approval_status': 'approved',
 'reviewer_comments': 'Looks good',
 'status': 'draft_created',
 'errors': [],
 'metadata': {}}

In [26]:
state["metadata"]["waiting_for_human"] = True

In [27]:
state["approval_status"] = "approved"

In [28]:
state["metadata"]["approval_history"] = [
    {"reviewer": "Manager", "decision": "approved"}
]